# 03 classical models

SARIMA, ETS and Prophet forecasting a month ahead (2015-06-30 to 2015-07-31), scored against the baselines from notebook 02. same split and same metrics (RMSE, MAE, WAPE).

built on store 1 first to check the pipeline, then scaled to a sample of stores.

## Set-Up

### Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import sklearn

import logging

sys.path.append('../src')
from load_clean_data import load_clean_data
from helpers import evaluate_forecast

In [2]:
df = load_clean_data()
df = df.sort_values('Date')
print(df.shape)

c:\Users\corbi\Documents\Projects\sales-forecasting\sales-forecasting-project\notebooks\../src\load_clean_data.py:6: DtypeWarning: Columns (7: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv('../data/train.csv')


(1017209, 18)


### Train Test Split

same split as notebook 02. train is everything before 2015-06-30, test is the 32 days from 2015-06-30.

In [3]:
train = df[df['Date'] < '2015-06-30']
test  = df[df['Date'] >= '2015-06-30']

length_check = len(train) + len(test) == len(df)

print(f"Length check: {length_check}")

Length check: True


## Single Store Models (Store 1)

### Extract Store 1 Data

In [4]:
store_1_train = train[train['Store'] == 1]
store_1_test  = test[test['Store'] == 1]

print(f"Store 1 train shape: {store_1_train.shape}")
print(f"Store 1 test shape: {store_1_test.shape}")

Store 1 train shape: (910, 18)
Store 1 test shape: (32, 18)


the models need a single series of sales with the date as the index. asfreq('D') sets the daily frequency, and the null checks confirm store 1 has no missing days.

In [5]:
store_1_train_series = store_1_train.set_index('Date')['Sales']
store_1_test_series  = store_1_test.set_index('Date')['Sales']

store_1_train_series = store_1_train_series.asfreq('D')
store_1_test_series = store_1_test_series.asfreq('D')

print(store_1_train_series.index)
print(store_1_test_series.index)
print(store_1_train_series.isnull().sum())
print(store_1_test_series.isnull().sum())

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
               '2013-01-09', '2013-01-10',
               ...
               '2015-06-20', '2015-06-21', '2015-06-22', '2015-06-23',
               '2015-06-24', '2015-06-25', '2015-06-26', '2015-06-27',
               '2015-06-28', '2015-06-29'],
              dtype='datetime64[us]', name='Date', length=910, freq='D')
DatetimeIndex(['2015-06-30', '2015-07-01', '2015-07-02', '2015-07-03',
               '2015-07-04', '2015-07-05', '2015-07-06', '2015-07-07',
               '2015-07-08', '2015-07-09', '2015-07-10', '2015-07-11',
               '2015-07-12', '2015-07-13', '2015-07-14', '2015-07-15',
               '2015-07-16', '2015-07-17', '2015-07-18', '2015-07-19',
               '2015-07-20', '2015-07-21', '2015-07-22', '2015-07-23',
               '2015-07-24', '2015-07-25', '2015-07-26', '2015-07-27',
               '2015-07-28', '2015-07-29

### SARIMA

order (1, 0, 1) and seasonal order (1, 1, 1, 7). d is 0 because store 1 has no trend, and 7 is the weekly cycle. forecast is clipped at 0 because sales can't be negative. first guess settings, not tuned.

#### Import

In [6]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

#### Model

In [7]:
sarima_model = SARIMAX(store_1_train_series, 
                       order=(1, 0, 1), 
                       seasonal_order=(1, 1, 1, 7))

sarima_results = sarima_model.fit()

In [8]:
sarima_forecast = sarima_results.forecast(steps=len(store_1_test_series)) #type: ignore

sarima_forecast = sarima_forecast.clip(lower=0)

length_check = len(sarima_forecast) == len(store_1_test_series)
print(f"Length check: {length_check}")

Length check: True


In [9]:
sarima_col = sarima_forecast.rename('SARIMA').rename_axis('Date').reset_index()

store_1_test = store_1_test.merge(sarima_col, on='Date', how='left')

print(store_1_test.shape)
print()
print(store_1_test.isnull().sum())
print()
print(store_1_test[['Store', 'DayOfWeek', 'Date', 'Sales', 'SARIMA']].head(10))

(32, 19)

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
SARIMA                       0
dtype: int64

   Store  DayOfWeek       Date  Sales       SARIMA
0      1          2 2015-06-30   5735  4609.887757
1      1          3 2015-07-01   5223  4435.338061
2      1          4 2015-07-02   5558  4041.689974
3      1          5 2015-07-03   4665  4515.724585
4      1          6 2015-07-04   4797  4954.005078
5      1          7 2015-07-05      0     0.000000
6      1          1 2015-07-06

#### Eval

In [10]:
RMSE, MAE, WAPE = evaluate_forecast(store_1_test_series, sarima_forecast)

RMSE: 701.03
MAE: 565.68
WAPE: 14.25%


### ETS 

additive seasonality with period 7 and no trend. additive because sunday is always 0, and multiplicative seasonality can't handle zeros.

#### Import

In [11]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

#### Model

In [12]:
ETS_model = ExponentialSmoothing(store_1_train_series,
                                 trend = None,
                                 seasonal = 'add',
                                 seasonal_periods = 7)

ETS_results = ETS_model.fit()

In [13]:
ETS_forecast = ETS_results.forecast(steps=len(store_1_test_series)) #type: ignore

ETS_forecast = ETS_forecast.clip(lower=0)

length_check = len(ETS_forecast) == len(store_1_test_series)
print(f"Length check: {length_check}")

Length check: True


In [14]:
ets_col = ETS_forecast.rename('ETS').rename_axis('Date').reset_index()

store_1_test = store_1_test.merge(ets_col, on='Date', how='left')

print(store_1_test.shape)
print()
print(store_1_test.isnull().sum())
print()
print(store_1_test[['Store', 'DayOfWeek', 'Date', 'Sales', 'ETS']].head(10))

(32, 20)

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
SARIMA                       0
ETS                          0
dtype: int64

   Store  DayOfWeek       Date  Sales          ETS
0      1          2 2015-06-30   5735  4212.551914
1      1          3 2015-07-01   5223  3990.242714
2      1          4 2015-07-02   5558  3645.118021
3      1          5 2015-07-03   4665  4137.169930
4      1          6 2015-07-04   4797  4556.925597
5      1          7 2015-07-05      0     0.000000

#### Eval

In [15]:
RMSE, MAE, WAPE= evaluate_forecast(store_1_test_series, ETS_forecast)

RMSE: 793.03
MAE: 608.60
WAPE: 15.33%


### Prophet

prophet takes a table with columns ds (date) and y (sales), and the dates are a normal column, not the index. default settings, forecast clipped at 0.

#### Import

In [ ]:
from prophet import Prophet
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

Importing plotly failed. Interactive plots will not work.


#### Making the ds / y table

In [17]:
prophet_ds_y_table = store_1_train[['Date', 'Sales']].rename(columns={'Date': 'ds', 'Sales': 'y'})

print(prophet_ds_y_table.head())
print(prophet_ds_y_table.shape[0])

                ds     y
1016095 2013-01-01     0
1014980 2013-01-02  5530
1013865 2013-01-03  4327
1012750 2013-01-04  4486
1011635 2013-01-05  4997
910


#### Model

In [18]:
prophet_model = Prophet()

prophet_model_fitted = prophet_model.fit(prophet_ds_y_table)

10:00:09 - cmdstanpy - INFO - Chain [1] start processing
10:00:09 - cmdstanpy - INFO - Chain [1] done processing


In [19]:
prophet_future_df = prophet_model_fitted.make_future_dataframe(periods=len(store_1_test_series), freq='D')

print(prophet_future_df.head())
print(prophet_future_df.tail())
print(prophet_future_df.shape[0])

          ds
0 2013-01-01
1 2013-01-02
2 2013-01-03
3 2013-01-04
4 2013-01-05
            ds
937 2015-07-27
938 2015-07-28
939 2015-07-29
940 2015-07-30
941 2015-07-31
942


In [20]:
prophet_preds = prophet_model_fitted.predict(prophet_future_df)

prophet_preds = prophet_preds.tail(len(store_1_test_series))

prophet_forecast = prophet_preds.set_index('ds')['yhat'].clip(lower=0)

In [21]:
prophet_col = prophet_forecast.rename('Prophet').rename_axis('Date').reset_index()

store_1_test = store_1_test.merge(prophet_col, on='Date', how='left')

print(store_1_test.shape)
print()
print(store_1_test.isnull().sum())
print()
print(store_1_test[['Store', 'DayOfWeek', 'Date', 'Sales', 'Prophet']].head(10))

(32, 21)

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
SARIMA                       0
ETS                          0
Prophet                      0
dtype: int64

   Store  DayOfWeek       Date  Sales      Prophet
0      1          2 2015-06-30   5735  4219.709178
1      1          3 2015-07-01   5223  4030.816088
2      1          4 2015-07-02   5558  3662.776560
3      1          5 2015-07-03   4665  4117.079554
4      1          6 2015-07-04   4797  4578.022782
5      1          7

#### Eval

In [22]:
RMSE, MAE, WAPE = evaluate_forecast(store_1_test_series, prophet_forecast)

RMSE: 766.94
MAE: 607.52
WAPE: 15.31%


### Single Store Model Scores:

| Model | RMSE | MAE | WAPE |
| --- | --- | --- | --- |
| SARIMA | 701.03 | 565.68 | 14.25% |
| ETS | 793.03 | 608.60 | 15.33% |
| Prophet | 766.94 | 607.52 | 15.31% |

### Export and Compare Single Store Scores

In [23]:
Models = ['SARIMA', 'ETS', 'Prophet']
rows   = []

for model in Models:
    print(f"Evaluating model: {model}")
    RMSE, MAE, WAPE = evaluate_forecast(store_1_test['Sales'], store_1_test[model])
    rows.append({ "Model": model, "RMSE": RMSE, "MAE": MAE, "WAPE": WAPE })
    print()

results_df = pd.DataFrame(rows)
results_df.to_csv('../results/single_store_classical_model_results.csv', index=False)

Evaluating model: SARIMA
RMSE: 701.03
MAE: 565.68
WAPE: 14.25%

Evaluating model: ETS
RMSE: 793.03
MAE: 608.60
WAPE: 15.33%

Evaluating model: Prophet
RMSE: 766.94
MAE: 607.52
WAPE: 15.31%



baseline rows come from notebook 02 (single_store_baseline_model_results.csv).

In [24]:
baseline_scores = pd.read_csv('../results/single_store_baseline_model_results.csv')
classical_scores = pd.read_csv('../results/single_store_classical_model_results.csv')

concatenated_scores = pd.concat([baseline_scores, classical_scores]).sort_values(by='WAPE', ascending=True)

concatenated_scores.style.hide(axis='index').format(precision=2)

Model,RMSE,MAE,WAPE
SARIMA,701.03,565.68,14.25
FullSeasonalAverage,702.58,567.18,14.29
Prophet,766.94,607.52,15.31
ETS,793.03,608.60,15.33
SeasonalAverage28Day,910.90,704.87,17.76
NaiveForecast,1029.52,797.59,20.10


| Model | RMSE | MAE | WAPE |
| --- | --- | --- | --- |
| SARIMA | 701.03 | 565.68 | 14.25% |
| Full Seasonal Average | 702.58 |567.18 | 14.29% |
| Prophet | 766.94 | 607.52 | 15.31% |
| ETS | 793.03 | 608.60 | 15.33% |
| 28 Day Seasonal Average | 910.90 | 704.87 | 17.76% |
| Naive | 1029.52 | 797.59 | 20.10% |

Finding: SARIMA (14.25%) and the full seasonal average (14.29%) are effectively tied. prophet and ETS are about a point behind (15.31%, 15.33%). the 28 day average and naive are clearly worse. the weekly pattern explains most of what is predictable here.

caveats:
- one store over one month, so small gaps could flip on another store or month
- none of the models are tuned
- none of the models see Promo

## Multi Store Models

### Store Information

In [25]:
def get_store_info(store_number, train=train):
    store = train[train['Store'] == store_number].set_index('Date')['Sales'].asfreq('D')

    store_nans = store[store.isnull()]

    has_gap = len(store_nans) > 0

    stretch_start = store_nans.index.max() + pd.Timedelta(days=1) if has_gap else store.index.min()

    stretch_days = (train['Date'].max() - stretch_start).days + 1

    return {'Store': store_number, 'has_gap' : has_gap, 'stretch_start': stretch_start, 'stretch_days': stretch_days}

In [26]:
print(get_store_info(1))

print(get_store_info(13))

{'Store': 1, 'has_gap': False, 'stretch_start': Timestamp('2013-01-01 00:00:00'), 'stretch_days': 910}
{'Store': 13, 'has_gap': True, 'stretch_start': Timestamp('2015-01-01 00:00:00'), 'stretch_days': 180}


In [27]:
store_nums = sorted(train['Store'].unique())
store_info_list = []

for store_num in store_nums:
    store_info = get_store_info(store_num)
    store_info_list.append(store_info)

store_info_df = pd.DataFrame(store_info_list)

In [28]:
print(store_info_df.shape)
print()
print(store_info_df.head(15))
print()
print(store_info_df['has_gap'].value_counts())
print()
print(store_info_df['stretch_start'].value_counts())

(1115, 4)

    Store  has_gap stretch_start  stretch_days
0       1    False    2013-01-01           910
1       2    False    2013-01-01           910
2       3    False    2013-01-01           910
3       4    False    2013-01-01           910
4       5    False    2013-01-01           910
5       6    False    2013-01-01           910
6       7    False    2013-01-01           910
7       8    False    2013-01-01           910
8       9    False    2013-01-01           910
9      10    False    2013-01-01           910
10     11    False    2013-01-01           910
11     12    False    2013-01-01           910
12     13     True    2015-01-01           180
13     14    False    2013-01-01           910
14     15    False    2013-01-01           910

has_gap
False    935
True     180
Name: count, dtype: int64

stretch_start
2013-01-01    934
2015-01-01    180
2013-01-02      1
Name: count, dtype: int64


### Model Functions

In [29]:
def get_store_series(store_number, mode, train=train, test=test):
    if mode == 'full':
        store_train_series = train[train['Store'] == store_number].set_index('Date')['Sales'].asfreq('D')
        store_test_series  = test[test['Store'] == store_number].set_index('Date')['Sales'].asfreq('D')
    elif mode == 'stretch':
        store_train_series = train[(train['Store'] == store_number) & (train['Date'] >= get_store_info(store_number)['stretch_start'])].set_index('Date')['Sales'].asfreq('D')
        store_test_series  = test[test['Store'] == store_number].set_index('Date')['Sales'].asfreq('D')
    else:
        raise ValueError("Invalid mode. Choose 'full' or 'stretch'.")

    return store_train_series, store_test_series

In [39]:
test_store_number = 13
practice_train_series, practice_test_series = get_store_series(test_store_number, 'full')

#### SARIMA

In [31]:
def forecast_SARIMA(train_series, steps):
    sarima_model = SARIMAX(train_series, 
                       order=(1, 0, 1), 
                       seasonal_order=(1, 1, 1, 7))

    sarima_results = sarima_model.fit(disp=False)

    sarima_forecast = sarima_results.forecast(steps=steps) #type: ignore

    sarima_forecast = sarima_forecast.clip(lower=0)

    return sarima_forecast

In [40]:
sarima_test = forecast_SARIMA(practice_train_series, len(practice_test_series))

evaluation_results = evaluate_forecast(practice_test_series, sarima_test)

RMSE: 1899.30
MAE: 1447.17
WAPE: 30.07%


#### ETS

In [33]:
def forecast_ETS(train_series, steps):
    ETS_model = ExponentialSmoothing(train_series,
                                 trend = None,
                                 seasonal = 'add',
                                 seasonal_periods = 7)

    ETS_results = ETS_model.fit()

    ETS_forecast = ETS_results.forecast(steps=steps) #type: ignore

    ETS_forecast = ETS_forecast.clip(lower=0)

    return ETS_forecast

In [34]:
ETS_test = forecast_ETS(practice_train_series, len(practice_test_series))
evaluation_results = evaluate_forecast(practice_test_series, ETS_test)

RMSE: 793.03
MAE: 608.60
WAPE: 15.33%


#### Prophet

In [35]:
def forecast_Prophet(train_series, steps):
    prophet_ds_y_table = train_series.reset_index().rename(columns={'Date': 'ds', 'Sales': 'y'})

    prophet_model = Prophet()

    prophet_model_fitted = prophet_model.fit(prophet_ds_y_table)

    prophet_future_df = prophet_model_fitted.make_future_dataframe(periods=steps, freq='D')

    prophet_preds = prophet_model_fitted.predict(prophet_future_df)

    prophet_preds = prophet_preds.tail(steps)

    prophet_forecast = prophet_preds.set_index('ds')['yhat'].clip(lower=0)

    return prophet_forecast.rename_axis('Date')

In [41]:
prophet_test = forecast_Prophet(practice_train_series, len(practice_test_series))
evaluation_results = evaluate_forecast(practice_test_series, prophet_test)

10:04:19 - cmdstanpy - INFO - Chain [1] start processing
10:04:19 - cmdstanpy - INFO - Chain [1] done processing


RMSE: 1964.60
MAE: 1596.71
WAPE: 33.18%


### Model Loop

In [43]:
no_gap_stores = store_info_df[store_info_df['has_gap'] == False]
gap_stores    = store_info_df[store_info_df['has_gap'] == True]

no_gap_stores = no_gap_stores[no_gap_stores['Store'] != 1]
gap_stores    = gap_stores[gap_stores['Store'] != 13]

no_gap_samples = no_gap_stores.sample(n=83, random_state=42)
gap_samples    = gap_stores.sample(n=15, random_state=42)

re_add_samples = store_info_df[store_info_df['Store'].isin([1, 13])]

sample_stores = pd.concat([no_gap_samples, gap_samples, re_add_samples]).sort_values(by='Store').reset_index(drop=True)

print(len(sample_stores))
print(sample_stores['Store'].is_unique)
print(sample_stores['has_gap'].value_counts())

100
True
has_gap
False    84
True     16
Name: count, dtype: int64


In [46]:
def forecast_store(store_number):
    store_train_series, store_test_series = get_store_series(store_number, 'stretch')
    steps = len(store_test_series)

    model_dict = {'SARIMA': forecast_SARIMA, 'ETS': forecast_ETS, 'Prophet': forecast_Prophet}

    forecasts = {}
    failed_models = []

    for model in model_dict.keys():
        try:
            print(f"Forecasting for Store {store_number} using {model}...")
            forecast_func = model_dict[model]
            forecast_series = forecast_func(store_train_series, steps)

            bad_nan   = forecast_series.isnull().any()
            bad_dates = not forecast_series.index.equals(store_test_series.index)

            if bad_nan or bad_dates:
                print(f"Forecast for Store {store_number} using {model} failed due to {'NaN values' if bad_nan else 'date mismatch'}")
                failed_models.append({'Store': store_number, 'Model': model,
                                'Reason': 'contains NaN' if bad_nan else 'dates do not match test period'})
            else:
                forecasts[model] = forecast_series

        except Exception as e:
            print(f"Error forecasting for Store {store_number} using {model}: {e}")
            failed_models.append({'Store': store_number, 'Model': model, 'Reason': str(e)})

    forecast_df = pd.DataFrame(forecasts).rename_axis('Date').reset_index()
    forecast_df.insert(0, 'Store', store_number)

    return forecast_df, failed_models       


In [47]:
print(forecast_store(1))

Forecasting for Store 1 using SARIMA...
Forecasting for Store 1 using ETS...
Forecasting for Store 1 using Prophet...


10:43:09 - cmdstanpy - INFO - Chain [1] start processing
10:43:09 - cmdstanpy - INFO - Chain [1] done processing


(    Store       Date       SARIMA          ETS      Prophet
0       1 2015-06-30  4609.887757  4212.551914  4219.709178
1       1 2015-07-01  4435.338061  3990.242714  4030.816088
2       1 2015-07-02  4041.689974  3645.118021  3662.776560
3       1 2015-07-03  4515.724585  4137.169930  4117.079554
4       1 2015-07-04  4954.005078  4556.925597  4578.022782
5       1 2015-07-05     0.000000     0.000000     0.000000
6       1 2015-07-06  4884.222571  4498.739398  4598.669638
7       1 2015-07-07  4621.345910  4212.551914  4323.929083
8       1 2015-07-08  4423.472517  3990.242714  4151.521833
9       1 2015-07-09  4045.931847  3645.118021  3797.890820
10      1 2015-07-10  4482.714640  4137.169930  4264.081385
11      1 2015-07-11  4927.448989  4556.925597  4734.009069
12      1 2015-07-12     0.000000     0.000000     0.000000
13      1 2015-07-13  4905.883496  4498.739398  4762.758273
14      1 2015-07-14  4626.696177  4212.551914  4486.767678
15      1 2015-07-15  4428.318583  3990